# Salifort Motors Employee Turnover Analysis

**Reproducible analytical case study**

This project examines workforce patterns associated with employee departure in the Salifort Motors educational HR dataset. It validates the raw data contract, documents the material class-balance change caused by exact-row deduplication, compares multiple classification models, tunes the operating threshold on validation data, and evaluates one selected model on a held-out test set.

The project is predictive and diagnostic—not causal. It is not a production HR scoring system and should not be used to automate employment decisions.

## 1. Reproducibility setup

The notebook uses only repository-relative paths. The verified source CSV is committed at `data/raw/HR_comma_sep.csv`. Restart the kernel and execute all cells from the repository root or from the `notebooks/` directory.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
import xgboost

from IPython.display import Markdown, display
from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
PRACTICAL_TIE_AP_TOLERANCE = 0.002

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "HR_comma_sep.csv"
IMAGES_DIR = PROJECT_ROOT / "images"
MODELS_DIR = PROJECT_ROOT / "models"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
sns.set_theme(context="notebook", style="whitegrid")

versions = {
    "python": platform.python_version(),
    "platform": platform.platform()
    "machine": platform.machine()
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}
display(pd.Series(versions, name="version").to_frame())
print(f"Data path: {DATA_PATH.relative_to(PROJECT_ROOT)}")

,version
python,3.13.5
pandas,2.2.3
numpy,2.3.5
scipy,1.17.0
matplotlib,3.10.8
seaborn,0.13.2
scikit_learn,1.8.0
xgboost,3.1.3


Data path: data/raw/HR_comma_sep.csv


## 2. Load and validate the raw dataset

The raw-file checks below are deliberately strict. A mismatched schema, row count, target distribution, missing-value count, duplicate count, or fingerprint will stop execution rather than silently analyzing the wrong file.

In [2]:
EXPECTED_COLUMNS = [
    "satisfaction_level",
    "last_evaluation",
    "number_project",
    "average_montly_hours",
    "time_spend_company",
    "Work_accident",
    "left",
    "promotion_last_5years",
    "Department",
    "salary",
]
EXPECTED_SHA256 = "af8c4cede39f28b5a67c748a66aa850fe260f908cbeaa226694121e0a9a4e105"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Required dataset not found at {DATA_PATH}. "
        "Place HR_comma_sep.csv in data/raw/ and rerun."
    )

raw_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df_raw = pd.read_csv(DATA_PATH)

raw_contract = {
    "rows": len(df_raw),
    "columns": df_raw.shape[1],
    "missing_values": int(df_raw.isna().sum().sum()),
    "exact_duplicate_rows": int(df_raw.duplicated().sum()),
    "stayed": int((df_raw["left"] == 0).sum()),
    "left": int((df_raw["left"] == 1).sum()),
    "departure_rate": float(df_raw["left"].mean()),
    "sha256": raw_sha256,
}

assert df_raw.shape == (14_999, 10), f"Unexpected shape: {df_raw.shape}"
assert df_raw.columns.tolist() == EXPECTED_COLUMNS, "Unexpected raw column schema"
assert raw_contract["missing_values"] == 0, "Unexpected missing values"
assert raw_contract["exact_duplicate_rows"] == 3_008, "Unexpected duplicate count"
assert raw_contract["stayed"] == 11_428 and raw_contract["left"] == 3_571
assert np.isclose(raw_contract["departure_rate"], 0.2380825388359224)
assert raw_sha256 == EXPECTED_SHA256, "Dataset SHA-256 fingerprint mismatch"

display(
    pd.Series(raw_contract, name="validated_value")
      .to_frame()
      .style.format({"validated_value": lambda x: f"{x:.4f}" if isinstance(x, float) else x})
)

,validated_value
rows,14999
columns,10
missing_values,0
exact_duplicate_rows,3008
stayed,11428
left,3571
departure_rate,0.2381
sha256,af8c4cede39f28b5a67c748a66aa850fe260f908cbeaa226694121e0a9a4e105


## 3. Clean column names and review duplicate impact

Legacy field names are standardized inside the analysis. Exact duplicates are removed as an explicit analytic assumption. Because the data has no employee ID or timestamp, duplicate provenance cannot be verified: duplicates may be extraction artifacts, but the supplied fields cannot prove that they are not legitimate repeated employee states.

This decision materially changes the class balance, so both versions are shown.

In [3]:
RENAME_MAP = {
    "number_project": "number_projects",
    "average_montly_hours": "average_monthly_hours",
    "time_spend_company": "tenure",
    "Work_accident": "work_accident",
    "Department": "department",
}

df_named = df_raw.rename(columns=RENAME_MAP).copy()
df = df_named.drop_duplicates().reset_index(drop=True)

balance_comparison = pd.DataFrame(
    {
        "rows": [len(df_named), len(df)],
        "stayed": [(df_named["left"] == 0).sum(), (df["left"] == 0).sum()],
        "left": [(df_named["left"] == 1).sum(), (df["left"] == 1).sum()],
        "departure_rate": [df_named["left"].mean(), df["left"].mean()],
    },
    index=["Raw data", "After exact-row deduplication"],
)
balance_comparison["percentage_point_change"] = (
    balance_comparison["departure_rate"] - balance_comparison.loc["Raw data", "departure_rate"]
) * 100

assert df.shape == (11_991, 10)
assert int((df["left"] == 0).sum()) == 10_000
assert int((df["left"] == 1).sum()) == 1_991
assert np.isclose(df["left"].mean(), 0.1660411975648403)

display(
    balance_comparison.style.format(
        {
            "rows": "{:,.0f}",
            "stayed": "{:,.0f}",
            "left": "{:,.0f}",
            "departure_rate": "{:.1%}",
            "percentage_point_change": "{:+.1f}",
        }
    )
)
print(
    f"Removing 3,008 exact duplicates lowers the observed departure rate "
    f"from {balance_comparison.loc['Raw data', 'departure_rate']:.1%} to "
    f"{balance_comparison.loc['After exact-row deduplication', 'departure_rate']:.1%}."
)

,rows,stayed,left,departure_rate,percentage_point_change
Raw data,"14,999","11,428","3,571",23.8%,+0.0
After exact-row deduplication,"11,991","10,000","1,991",16.6%,-7.2


Removing 3,008 exact duplicates lowers the observed departure rate from 23.8% to 16.6%.


### Data-quality interpretation

- No missing values were found.
- Exact-row deduplication removes 20.1% of raw rows.
- The positive-class rate falls by about 7.2 percentage points, from 23.8% to 16.6%.
- The analysis therefore reports results only for the deduplicated analytic sample and preserves the raw-versus-clean comparison.
- No row-level employee identifiers or timestamps are available, so repeated-person structure, temporal ordering, and duplicate provenance remain unverifiable.

## 4. Exploratory analysis

The exploratory analysis focuses on seven visuals that most directly support the business question and model interpretation.

In [4]:
project_turnover = (
    df.groupby("number_projects")["left"]
      .agg(turnover_rate="mean", employees="size")
)
tenure_turnover = (
    df.groupby("tenure")["left"]
      .agg(turnover_rate="mean", employees="size")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(project_turnover.index.astype(str), project_turnover["turnover_rate"])
axes[0].set(
    title="Departure Rate by Project Count",
    xlabel="Number of projects",
    ylabel="Departure rate",
    ylim=(0, 1),
)
axes[1].bar(tenure_turnover.index.astype(str), tenure_turnover["turnover_rate"])
axes[1].set(
    title="Departure Rate by Tenure",
    xlabel="Years at company",
    ylabel="Departure rate",
    ylim=(0, 1),
)
fig.suptitle("Workload and tenure show strongly nonlinear departure patterns", y=1.02)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "turnover_by_projects_tenure.png", dpi=160, bbox_inches="tight")
plt.show()

display(project_turnover.style.format({"turnover_rate": "{:.1%}", "employees": "{:,.0f}"}))
display(tenure_turnover.style.format({"turnover_rate": "{:.1%}", "employees": "{:,.0f}"}))

,turnover_rate,employees
number_projects,,
2,54.2%,"1,582"
3,1.1%,"3,520"
4,6.4%,"3,685"
5,15.4%,"2,233"
6,44.9%,826
7,100.0%,145


,turnover_rate,employees
tenure,,
2,1.1%,"2,910"
3,16.8%,"5,190"
4,24.7%,"2,005"
5,45.4%,"1,062"
6,20.1%,542
7,0.0%,94
8,0.0%,81
10,0.0%,107


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(
    data=df,
    x="satisfaction_level",
    hue="left",
    bins=25,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[0],
)
axes[0].set_title("Satisfaction Distribution by Outcome")
axes[0].set_xlabel("Satisfaction level")

sns.histplot(
    data=df,
    x="average_monthly_hours",
    hue="left",
    bins=25,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1],
)
axes[1].set_title("Monthly Hours Distribution by Outcome")
axes[1].set_xlabel("Average monthly hours")

fig.tight_layout()
fig.savefig(IMAGES_DIR / "satisfaction_and_hours.png", dpi=160, bbox_inches="tight")
plt.show()

In [6]:
salary_order = ["low", "medium", "high"]
salary_turnover = (
    df.groupby("salary")["left"]
      .agg(turnover_rate="mean", employees="size")
      .reindex(salary_order)
)
department_turnover = (
    df.groupby("department")["left"]
      .agg(turnover_rate="mean", employees="size")
      .sort_values("turnover_rate")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(salary_turnover.index, salary_turnover["turnover_rate"])
axes[0].set(title="Departure Rate by Salary Band", ylabel="Departure rate", ylim=(0, 0.25))
axes[1].barh(department_turnover.index, department_turnover["turnover_rate"])
axes[1].set(title="Departure Rate by Department", xlabel="Departure rate")
fig.tight_layout()
fig.savefig(IMAGES_DIR / "turnover_by_salary_department.png", dpi=160, bbox_inches="tight")
plt.show()

display(salary_turnover.style.format({"turnover_rate": "{:.1%}", "employees": "{:,.0f}"}))
display(department_turnover.style.format({"turnover_rate": "{:.1%}", "employees": "{:,.0f}"}))

,turnover_rate,employees
salary,,
low,20.5%,"5,740"
medium,14.6%,"5,261"
high,4.8%,990


,turnover_rate,employees
department,,
management,11.9%,436
RandD,12.2%,694
product_mng,16.0%,686
IT,16.2%,976
marketing,16.6%,673
sales,17.0%,"3,239"
support,17.1%,"1,821"
technical,17.4%,"2,244"
accounting,17.6%,621


In [7]:
left_satisfaction = df.loc[df["left"] == 1, "satisfaction_level"]
stayed_satisfaction = df.loc[df["left"] == 0, "satisfaction_level"]

t_stat, t_p = stats.ttest_ind(left_satisfaction, stayed_satisfaction, equal_var=False)
pooled_sd = np.sqrt(
    (
        (len(left_satisfaction) - 1) * left_satisfaction.var(ddof=1)
        + (len(stayed_satisfaction) - 1) * stayed_satisfaction.var(ddof=1)
    )
    / (len(left_satisfaction) + len(stayed_satisfaction) - 2)
)
cohens_d = (left_satisfaction.mean() - stayed_satisfaction.mean()) / pooled_sd

salary_table = pd.crosstab(df["salary"], df["left"])
chi2, chi_p, _, _ = chi2_contingency(salary_table)
cramers_v = np.sqrt(
    chi2
    / (
        salary_table.to_numpy().sum()
        * min(salary_table.shape[0] - 1, salary_table.shape[1] - 1)
    )
)

statistical_checks = pd.DataFrame(
    {
        "analysis": [
            "Welch t-test: satisfaction",
            "Cohen's d: satisfaction",
            "Chi-square: salary and outcome",
            "Cramér's V: salary and outcome",
        ],
        "estimate": [t_stat, cohens_d, chi2, cramers_v],
        "p_value": [t_p, np.nan, chi_p, np.nan],
    }
)
display(statistical_checks.style.format({"estimate": "{:.4f}", "p_value": "{:.3e}"}))

,analysis,estimate,p_value
0,Welch t-test: satisfaction,-35.8893,1.194e-228
1,Cohen's d: satisfaction,-1.0058,nan
2,Chi-square: salary and outcome,175.2107,8.984e-39
3,Cramér's V: salary and outcome,0.1209,nan


### Exploratory findings

The deduplicated sample suggests two broad risk profiles: under-engaged employees with very low project loads and dissatisfied employees carrying high workloads. Departure is especially concentrated at project-load extremes, around the 4–5 year tenure window, at lower satisfaction levels, and among lower-salary records. Department differences exist, but are smaller than the strongest workload, tenure, satisfaction, and salary patterns.

These are associations within the supplied educational dataset. They do not prove why an employee left.

## 5. Modeling design

### Honest feature-set definitions

- **Base feature set:** the nine cleaned source predictors exactly once. `salary` remains categorical and is one-hot encoded inside the pipeline.
- **Engineered feature set:** the base fields plus row-wise workload, tenure, and interaction features. No target aggregates are used.
- `salary_numeric` is not created or modeled, eliminating duplicate salary encoding.

Preprocessing is fitted inside each scikit-learn pipeline after splitting. Threshold selection uses validation data only. The held-out test set is reserved for the selected champion.

In [8]:
BASE_FEATURES = [
    "satisfaction_level",
    "last_evaluation",
    "number_projects",
    "average_monthly_hours",
    "tenure",
    "work_accident",
    "promotion_last_5years",
    "department",
    "salary",
]

def add_engineered_features(frame):
    out = frame.copy()
    standard_month = 40 * 50 / 12
    out["overtime_hours_monthly"] = (
        out["average_monthly_hours"] - standard_month
    ).clip(lower=0)
    out["hours_per_project"] = (
        out["average_monthly_hours"] / out["number_projects"].clip(lower=1)
    )
    out["project_load_extreme"] = (
        (out["number_projects"] <= 2) | (out["number_projects"] >= 6)
    ).astype(int)
    out["high_hours"] = (out["average_monthly_hours"] >= 220).astype(int)
    out["low_hours"] = (out["average_monthly_hours"] <= 160).astype(int)
    out["career_stagnation"] = (
        (out["tenure"] >= 4) & (out["promotion_last_5years"] == 0)
    ).astype(int)
    out["high_performer_overworked"] = (
        (out["last_evaluation"] >= 0.80)
        & (out["average_monthly_hours"] >= 220)
    ).astype(int)
    out["low_satisfaction"] = (out["satisfaction_level"] < 0.40).astype(int)
    out["evaluation_satisfaction_gap"] = (
        out["last_evaluation"] - out["satisfaction_level"]
    )
    out["satisfaction_evaluation_interaction"] = (
        out["satisfaction_level"] * out["last_evaluation"]
    )
    out["workload_interaction"] = (
        out["number_projects"] * out["average_monthly_hours"]
    )
    out["tenure_squared"] = out["tenure"] ** 2
    out["tenure_band"] = pd.cut(
        out["tenure"],
        bins=[0, 2, 4, 6, np.inf],
        labels=["0-2", "3-4", "5-6", "7+"],
        include_lowest=True,
    )
    return out

X_base = df[BASE_FEATURES].copy()
X_engineered = add_engineered_features(X_base)
y = df["left"].astype(int).copy()

engineered_only = [column for column in X_engineered.columns if column not in X_base.columns]
feature_definition = pd.DataFrame(
    {
        "feature_set": ["Base", "Engineered"],
        "predictor_count": [X_base.shape[1], X_engineered.shape[1]],
        "salary_representation": ["Categorical only", "Categorical only"],
        "additional_features": ["None", ", ".join(engineered_only)],
    }
)
display(feature_definition)

,feature_set,predictor_count,salary_representation,additional_features
0,Base,9,Categorical only,None
1,Engineered,22,Categorical only,"overtime_hours_monthly, hours_per_project, pro..."


In [9]:
train_idx, remainder_idx = train_test_split(
    df.index,
    test_size=0.40,
    stratify=y,
    random_state=RANDOM_STATE,
)
validation_idx, test_idx = train_test_split(
    remainder_idx,
    test_size=0.50,
    stratify=y.loc[remainder_idx],
    random_state=RANDOM_STATE,
)

Xb_train, Xb_validation, Xb_test = (
    X_base.loc[train_idx],
    X_base.loc[validation_idx],
    X_base.loc[test_idx],
)
Xe_train, Xe_validation, Xe_test = (
    X_engineered.loc[train_idx],
    X_engineered.loc[validation_idx],
    X_engineered.loc[test_idx],
)
y_train, y_validation, y_test = (
    y.loc[train_idx],
    y.loc[validation_idx],
    y.loc[test_idx],
)

split_summary = pd.DataFrame(
    {
        "rows": [len(train_idx), len(validation_idx), len(test_idx)],
        "departure_rate": [y_train.mean(), y_validation.mean(), y_test.mean()],
    },
    index=["Training", "Validation", "Test"],
)
display(split_summary.style.format({"rows": "{:,.0f}", "departure_rate": "{:.1%}"}))

,rows,departure_rate
Training,"7,194",16.6%
Validation,"2,398",16.6%
Test,"2,399",16.6%


In [10]:
def build_preprocessor(frame, scale_numeric=False):
    categorical_columns = frame.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_columns = [column for column in frame.columns if column not in categorical_columns]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric_columns),
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )

def classification_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "f2": float(fbeta_score(y_true, predictions, beta=2, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "average_precision": float(average_precision_score(y_true, probabilities)),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
    }

def best_f2_threshold(y_true, probabilities):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [
        fbeta_score(y_true, probabilities >= threshold, beta=2, zero_division=0)
        for threshold in thresholds
    ]
    best_index = int(np.argmax(scores))
    return float(thresholds[best_index]), float(scores[best_index])

### Candidate models

The comparison includes a dummy baseline, two logistic-regression specifications, a decision tree, a random forest, and base/engineered XGBoost variants. F2 is used for threshold tuning because missed departures are treated as more costly than additional false-positive reviews, while average precision is the primary ranking metric for the imbalanced target.

In [11]:
xgb_settings = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "n_estimators": 201,
    "max_depth": 6,
    "learning_rate": 0.02894,
    "min_child_weight": 2,
    "subsample": 0.8574,
    "colsample_bytree": 0.7545,
    "reg_lambda": 0.5408,
    "reg_alpha": 0.00105,
}

models = {
    "Dummy baseline": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xb_train)),
                ("model", DummyClassifier(strategy="prior")),
            ]
        ),
        "base",
    ),
    "Logistic regression — base": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xb_train, scale_numeric=True)),
                (
                    "model",
                    LogisticRegression(
                        C=0.1,
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "base",
    ),
    "Logistic regression — engineered": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xe_train, scale_numeric=True)),
                (
                    "model",
                    LogisticRegression(
                        C=1.0,
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "engineered",
    ),
    "Decision tree — engineered": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xe_train)),
                (
                    "model",
                    DecisionTreeClassifier(
                        min_samples_leaf=20,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "engineered",
    ),
    "Random forest — engineered": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xe_train)),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=500,
                        max_depth=8,
                        min_samples_leaf=2,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "engineered",
    ),
    "XGBoost — base": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xb_train)),
                ("model", XGBClassifier(**xgb_settings)),
            ]
        ),
        "base",
    ),
    "XGBoost — engineered": (
        Pipeline(
            [
                ("preprocess", build_preprocessor(Xe_train)),
                ("model", XGBClassifier(**xgb_settings)),
            ]
        ),
        "engineered",
    ),
}

In [12]:
validation_rows = []
validation_probabilities = {}

for model_name, (model, feature_set) in models.items():
    X_train_model, X_validation_model = (
        (Xb_train, Xb_validation)
        if feature_set == "base"
        else (Xe_train, Xe_validation)
    )
    start = time.perf_counter()
    model.fit(X_train_model, y_train)
    fit_seconds = time.perf_counter() - start

    probabilities = model.predict_proba(X_validation_model)[:, 1]
    threshold, _ = best_f2_threshold(y_validation, probabilities)
    row = classification_metrics(y_validation, probabilities, threshold)
    row.update(
        {
            "model": model_name,
            "feature_set": feature_set,
            "fit_seconds": fit_seconds,
        }
    )
    validation_rows.append(row)
    validation_probabilities[model_name] = probabilities

validation_results = (
    pd.DataFrame(validation_rows)
      .set_index("model")
      .sort_values(["average_precision", "f2"], ascending=False)
)

display(
    validation_results.style.format(
        {
            "threshold": "{:.3f}",
            "accuracy": "{:.4f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1": "{:.4f}",
            "f2": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "brier_score": "{:.4f}",
            "fit_seconds": "{:.2f}",
        }
    )
)

,threshold,accuracy,precision,recall,f1,f2,roc_auc,average_precision,brier_score,feature_set,fit_seconds
model,,,,,,,,,,,
XGBoost — engineered,0.460,0.9867,0.9919,0.9271,0.9584,0.9394,0.9877,0.9710,0.0137,engineered,0.28
XGBoost — base,0.265,0.9817,0.9562,0.9322,0.9440,0.9369,0.9870,0.9698,0.0150,base,0.16
Random forest — engineered,0.370,0.9800,0.9464,0.9322,0.9392,0.9350,0.9860,0.9692,0.0184,engineered,2.61
Decision tree — engineered,0.560,0.9725,0.9029,0.9347,0.9185,0.9281,0.9719,0.9436,0.0344,engineered,0.06
Logistic regression — engineered,0.505,0.9153,0.6816,0.9196,0.7829,0.8596,0.9561,0.8476,0.0705,engineered,0.07
Logistic regression — base,0.440,0.7494,0.3881,0.8844,0.5395,0.7043,0.8368,0.3718,0.1712,base,0.10
Dummy baseline,0.050,0.1660,0.1660,1.0000,0.2847,0.4987,0.5000,0.1660,0.1384,base,0.02


In [13]:
plot_metrics = validation_results[["average_precision", "f2", "recall", "precision"]]
ax = plot_metrics.plot(kind="bar", figsize=(12, 6))
ax.set(
    title="Validation Performance at Each Model's F2-Optimized Threshold",
    ylabel="Score",
    ylim=(0, 1.05),
)
plt.xticks(rotation=32, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "validation_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

### Practical-tie model selection

A difference smaller than **0.002 average-precision points** is treated as a practical tie rather than evidence of a meaningfully better model. Among tied leaders, the analysis prefers a model using the base source features because it requires fewer engineered assumptions and is easier to explain and maintain. F2 and other metrics are reported, but the held-out test set is not used to break the tie.

In [14]:
best_validation_ap = validation_results["average_precision"].max()
tie_candidates = validation_results[
    validation_results["average_precision"]
    >= best_validation_ap - PRACTICAL_TIE_AP_TOLERANCE
].copy()

base_tie_candidates = tie_candidates[tie_candidates["feature_set"] == "base"]
selection_pool = base_tie_candidates if not base_tie_candidates.empty else tie_candidates
champion_name = selection_pool.sort_values(
    ["average_precision", "f2"], ascending=False
).index[0]

champion_model, champion_feature_set = models[champion_name]
champion_threshold = float(validation_results.loc[champion_name, "threshold"])

display(
    tie_candidates[["feature_set", "average_precision", "f2", "precision", "recall", "threshold"]]
    .style.format(
        {
            "average_precision": "{:.4f}",
            "f2": "{:.4f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "threshold": "{:.3f}",
        }
    )
)
print(f"Selected champion: {champion_name}")
print(f"Feature set: {champion_feature_set}")
print(f"Validation-selected threshold: {champion_threshold:.3f}")

,feature_set,average_precision,f2,precision,recall,threshold
model,,,,,,
XGBoost — engineered,engineered,0.9710,0.9394,0.9919,0.9271,0.460
XGBoost — base,base,0.9698,0.9369,0.9562,0.9322,0.265
Random forest — engineered,engineered,0.9692,0.9350,0.9464,0.9322,0.370


Selected champion: XGBoost — base
Feature set: base
Validation-selected threshold: 0.265


## 6. Held-out test evaluation

Only the selected champion is evaluated below. The threshold remains fixed at the value selected on the validation split.

In [15]:
X_test_champion = Xb_test if champion_feature_set == "base" else Xe_test
test_probabilities = champion_model.predict_proba(X_test_champion)[:, 1]
test_predictions = (test_probabilities >= champion_threshold).astype(int)

test_metrics = classification_metrics(
    y_test,
    test_probabilities,
    champion_threshold,
)
test_metrics_table = pd.Series(test_metrics, name="test_value").to_frame()
display(test_metrics_table.style.format({"test_value": "{:.4f}"}))

confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
tn, fp, fn, tp = [int(value) for value in confusion.ravel()]
print(classification_report(y_test, test_predictions, target_names=["Stayed", "Left"], digits=3))
print(
    f"True negatives: {tn:,} | False positives: {fp:,} | "
    f"False negatives: {fn:,} | True positives: {tp:,}"
)

,test_value
threshold,0.2650
accuracy,0.9817
precision,0.9515
recall,0.9372
f1,0.9443
f2,0.9400
roc_auc,0.9839
average_precision,0.9679
brier_score,0.0146


              precision    recall  f1-score   support

      Stayed      0.988     0.991     0.989      2001
        Left      0.952     0.937     0.944       398

    accuracy                          0.982      2399
   macro avg      0.970     0.964     0.967      2399
weighted avg      0.982     0.982     0.982      2399

True negatives: 1,982 | False positives: 19 | False negatives: 25 | True positives: 373


In [16]:
fpr, tpr, _ = roc_curve(y_test, test_probabilities)
pr_precision, pr_recall, _ = precision_recall_curve(y_test, test_probabilities)
fraction_positive, mean_predicted = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="quantile",
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Stayed", "Left"],
).plot(ax=axes[0], values_format="d", colorbar=False)
axes[0].set_title("Confusion Matrix")

axes[1].plot(
    pr_recall,
    pr_precision,
    label=f"Average precision = {test_metrics['average_precision']:.3f}",
)
axes[1].axhline(y_test.mean(), linestyle="--", label="Class prevalence")
axes[1].set(
    title="Precision–Recall Curve",
    xlabel="Recall",
    ylabel="Precision",
)
axes[1].legend()

axes[2].plot(mean_predicted, fraction_positive, marker="o", label="Observed")
axes[2].plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
axes[2].set(
    title=f"Calibration (Brier = {test_metrics['brier_score']:.3f})",
    xlabel="Mean predicted probability",
    ylabel="Observed departure rate",
)
axes[2].legend()

fig.tight_layout()
fig.savefig(IMAGES_DIR / "test_performance_summary.png", dpi=160, bbox_inches="tight")
plt.show()

In [17]:
def bootstrap_intervals(
    y_true,
    probabilities,
    threshold,
    iterations=500,
    random_state=RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    y_array = np.asarray(y_true)
    probability_array = np.asarray(probabilities)
    samples = []

    for _ in range(iterations):
        indices = rng.integers(0, len(y_array), len(y_array))
        sampled_y = y_array[indices]
        sampled_probabilities = probability_array[indices]
        if np.unique(sampled_y).size == 2:
            samples.append(
                classification_metrics(
                    sampled_y,
                    sampled_probabilities,
                    threshold,
                )
            )

    bootstrap_results = pd.DataFrame(samples)
    rows = []
    for metric in [
        "precision",
        "recall",
        "f1",
        "f2",
        "roc_auc",
        "average_precision",
        "brier_score",
    ]:
        rows.append(
            {
                "metric": metric,
                "estimate": test_metrics[metric],
                "lower_95": bootstrap_results[metric].quantile(0.025),
                "upper_95": bootstrap_results[metric].quantile(0.975),
            }
        )
    return pd.DataFrame(rows).set_index("metric")

bootstrap_summary = bootstrap_intervals(
    y_test,
    test_probabilities,
    champion_threshold,
)
display(
    bootstrap_summary.style.format(
        {"estimate": "{:.4f}", "lower_95": "{:.4f}", "upper_95": "{:.4f}"}
    )
)

,estimate,lower_95,upper_95
metric,,,
precision,0.9515,0.9306,0.9711
recall,0.9372,0.9119,0.9579
f1,0.9443,0.9280,0.9593
f2,0.9400,0.9180,0.9582
roc_auc,0.9839,0.9754,0.9911
average_precision,0.9679,0.9535,0.9789
brier_score,0.0146,0.0107,0.0190


## 7. Interpretability and subgroup review

Permutation importance is computed on the held-out test features using average precision as the scoring function. Salary- and department-level error tables are descriptive subgroup checks, not a fairness audit. Protected-class bias cannot be assessed because protected attributes are not supplied.

In [18]:
permutation = permutation_importance(
    champion_model,
    X_test_champion,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance = (
    pd.DataFrame(
        {
            "feature": X_test_champion.columns,
            "importance_mean": permutation.importances_mean,
            "importance_std": permutation.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance.head(12).style.format({"importance_mean": "{:.4f}", "importance_std": "{:.4f}"}))

plot_data = importance.head(10).sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(
    plot_data["feature"],
    plot_data["importance_mean"],
    xerr=plot_data["importance_std"],
)
ax.set(
    title="Permutation Importance on the Held-Out Test Set",
    xlabel="Reduction in average precision after permutation",
)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "permutation_importance.png", dpi=160, bbox_inches="tight")
plt.show()

,feature,importance_mean,importance_std
0,satisfaction_level,0.3722,0.0104
1,tenure,0.1149,0.0040
2,number_projects,0.0618,0.0041
3,average_monthly_hours,0.0392,0.0052
4,last_evaluation,0.0346,0.0035
5,work_accident,0.0006,0.0008
6,promotion_last_5years,0.0000,0.0000
7,salary,-0.0000,0.0011
8,department,-0.0005,0.0003


In [19]:
def subgroup_error_table(raw_features, group_column):
    work = pd.DataFrame(
        {
            group_column: raw_features[group_column].to_numpy(),
            "actual": np.asarray(y_test),
            "probability": test_probabilities,
        }
    )
    work["prediction"] = (work["probability"] >= champion_threshold).astype(int)
    rows = []

    for group_value, group in work.groupby(group_column):
        tn_group, fp_group, fn_group, tp_group = confusion_matrix(
            group["actual"],
            group["prediction"],
            labels=[0, 1],
        ).ravel()
        rows.append(
            {
                "group": group_value,
                "n": len(group),
                "observed_departure_rate": group["actual"].mean(),
                "predicted_positive_rate": group["prediction"].mean(),
                "precision": precision_score(
                    group["actual"], group["prediction"], zero_division=0
                ),
                "recall": recall_score(
                    group["actual"], group["prediction"], zero_division=0
                ),
                "false_positive_rate": (
                    fp_group / (fp_group + tn_group)
                    if (fp_group + tn_group)
                    else np.nan
                ),
            }
        )
    return pd.DataFrame(rows).sort_values("group")

raw_test = X_base.loc[test_idx]
salary_subgroups = subgroup_error_table(raw_test, "salary")
department_subgroups = subgroup_error_table(raw_test, "department")

percentage_columns = [
    "observed_departure_rate",
    "predicted_positive_rate",
    "precision",
    "recall",
    "false_positive_rate",
]
display(
    salary_subgroups.style.format(
        {column: "{:.1%}" for column in percentage_columns}
    )
)
display(
    department_subgroups.style.format(
        {column: "{:.1%}" for column in percentage_columns}
    )
)

,group,n,observed_departure_rate,predicted_positive_rate,precision,recall,false_positive_rate
0,high,178,6.2%,6.2%,100.0%,100.0%,0.0%
1,low,1157,19.4%,19.7%,93.9%,95.1%,1.5%
2,medium,1064,15.2%,14.4%,96.7%,91.4%,0.6%


,group,n,observed_departure_rate,predicted_positive_rate,precision,recall,false_positive_rate
0,IT,190,12.1%,12.1%,100.0%,100.0%,0.0%
1,RandD,144,10.4%,9.7%,92.9%,86.7%,0.8%
2,accounting,120,18.3%,16.7%,100.0%,90.9%,0.0%
3,hr,123,17.9%,16.3%,100.0%,90.9%,0.0%
4,management,88,9.1%,10.2%,77.8%,87.5%,2.5%
5,marketing,153,14.4%,13.7%,100.0%,95.5%,0.0%
6,product_mng,135,18.5%,19.3%,96.2%,100.0%,0.9%
7,sales,657,17.4%,17.5%,93.0%,93.9%,1.5%
8,support,343,20.1%,19.0%,100.0%,94.2%,0.0%
9,technical,446,17.5%,17.7%,91.1%,92.3%,1.9%


In [20]:
profile = raw_test.copy()
profile["actual_left"] = y_test.to_numpy()
profile["predicted_risk"] = test_probabilities

profile_groups = {
    "Satisfaction quintile": pd.qcut(
        profile["satisfaction_level"], 5, duplicates="drop"
    ),
    "Monthly-hours quintile": pd.qcut(
        profile["average_monthly_hours"], 5, duplicates="drop"
    ),
    "Tenure": profile["tenure"].astype(str),
    "Project count": profile["number_projects"].astype(str),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (title, grouping) in zip(axes.flat, profile_groups.items()):
    grouped = (
        profile.assign(group=grouping)
        .groupby("group", observed=False)[["actual_left", "predicted_risk"]]
        .mean()
    )
    positions = np.arange(len(grouped))
    ax.plot(positions, grouped["actual_left"], marker="o", label="Observed")
    ax.plot(positions, grouped["predicted_risk"], marker="o", label="Predicted")
    ax.set_xticks(positions)
    ax.set_xticklabels(grouped.index.astype(str), rotation=28, ha="right")
    ax.set_title(title)
    ax.set_ylabel("Rate")
    ax.legend()

fig.suptitle("Observed versus predicted departure rates in aggregate groups", y=1.01)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "observed_vs_predicted_profiles.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Credibility, limitations, and ethical use

The very high ranking performance is credible as a result **within this educational dataset**, but it is not evidence that a real-world employee-turnover system would generalize similarly. The data has unusually strong separation, lacks dates and employee identifiers, combines voluntary and involuntary exits, and does not establish when satisfaction or evaluation measures were collected relative to departure. A random split cannot substitute for future-period validation.

The output should support aggregate investigation, policy review, staffing analysis, and prospective testing of supportive interventions. It should not rank employees for adverse action, automate employment decisions, or be presented as causal evidence. Salary and department error tables are limited descriptive checks and are not substitutes for protected-class bias assessment.

## 9. Business recommendations

1. Investigate teams and roles with sustained project-load extremes rather than assuming a single ideal workload for everyone.
2. Review high monthly-hours patterns alongside staffing, schedule design, and manager practices.
3. Examine the 4–6 year tenure window for career progression, compensation, promotion access, and role mobility.
4. Compare compensation and promotion patterns within genuinely comparable roles rather than using salary band alone.
5. Use aggregate monitoring and employee feedback to design supportive interventions, then test whether those interventions improve retention prospectively.
6. Keep individual predictions restricted, reviewable, and non-decisional; public portfolio artifacts should show aggregate results only.

In [21]:
metadata = {
    "reproduction_status": "Fresh-kernel rerun passed on 2026-07-30",
    "random_state": RANDOM_STATE,
    "data": {
        "repository_path": str(DATA_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "upstream_dataset": "Hr Analytics Job Prediction",
        "upstream_publisher": "Faisal Qureshi",
        "upstream_source": "https://www.kaggle.com/datasets/mfaisalqureshi/hr-analytics-and-job-prediction",
        "upstream_license": "CC0 1.0 Universal",
        "raw_csv_committed": True,
        "required_filename": DATA_PATH.name,
        "sha256": raw_sha256,
        "raw_shape": list(df_raw.shape),
        "raw_exact_duplicates": int(df_raw.duplicated().sum()),
        "raw_target_counts": {
            "stayed": int((df_raw["left"] == 0).sum()),
            "left": int((df_raw["left"] == 1).sum()),
        },
        "raw_departure_rate": float(df_raw["left"].mean()),
        "deduplicated_shape": list(df.shape),
        "deduplicated_target_counts": {
            "stayed": int((df["left"] == 0).sum()),
            "left": int((df["left"] == 1).sum()),
        },
        "deduplicated_departure_rate": float(df["left"].mean()),
    },
    "split": {
        name.lower(): {
            "rows": int(split_summary.loc[name, "rows"]),
            "departure_rate": float(split_summary.loc[name, "departure_rate"]),
        }
        for name in split_summary.index
    },
    "feature_sets": {
        "base": X_base.columns.tolist(),
        "engineered": X_engineered.columns.tolist(),
        "engineered_only": engineered_only,
        "salary_encoding": "salary retained once as a categorical feature; salary_numeric not used",
    },
    "selection": {
        "primary_metric": "average_precision",
        "threshold_objective": "F2 on validation split",
        "practical_tie_tolerance_average_precision": PRACTICAL_TIE_AP_TOLERANCE,
        "tie_candidates": tie_candidates.index.tolist(),
        "tie_break_rule": "Prefer a tied base-feature model; then higher validation average precision and F2",
        "champion": champion_name,
        "feature_set": champion_feature_set,
        "threshold": champion_threshold,
    },
    "validation_results": {
        model_name: {
            column: (
                float(value)
                if isinstance(value, (int, float, np.integer, np.floating))
                else value
            )
            for column, value in row.items()
        }
        for model_name, row in validation_results.to_dict(orient="index").items()
    },
    "test_metrics": test_metrics,
    "confusion_matrix": {
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    },
    "bootstrap_95_intervals": {
        metric: {
            "estimate": float(row["estimate"]),
            "lower_95": float(row["lower_95"]),
            "upper_95": float(row["upper_95"]),
        }
        for metric, row in bootstrap_summary.iterrows()
    },
    "top_permutation_importance": [
        {
            "feature": row["feature"],
            "importance_mean": float(row["importance_mean"]),
            "importance_std": float(row["importance_std"]),
        }
        for _, row in importance.head(10).iterrows()
    ],
    "versions": versions,
    "model_binary_committed": False,
}

metadata_path = MODELS_DIR / "salifort_turnover_model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved metadata: {metadata_path.relative_to(PROJECT_ROOT)}")
print("No binary model file was written to the public package.")

Saved metadata: models/salifort_turnover_model_metadata.json
No binary model file was written to the public package.


In [22]:
final_summary = f'''
## Final test result

The selected champion is **{champion_name}** using the **{champion_feature_set}** feature set. Its operating threshold, selected on validation data by maximizing F2, is **{champion_threshold:.3f}**.

Held-out test results:

- Accuracy: **{test_metrics["accuracy"]:.1%}**
- Precision: **{test_metrics["precision"]:.1%}**
- Recall: **{test_metrics["recall"]:.1%}**
- F1: **{test_metrics["f1"]:.1%}**
- F2: **{test_metrics["f2"]:.1%}**
- ROC-AUC: **{test_metrics["roc_auc"]:.4f}**
- Average precision: **{test_metrics["average_precision"]:.4f}**
- Brier score: **{test_metrics["brier_score"]:.4f}**

Confusion-matrix counts: **{tn} true negatives, {fp} false positives, {fn} false negatives, and {tp} true positives**.

Reproduction status: **Fresh-kernel rerun passed on 2026-07-30**
'''
display(Markdown(final_summary))


## Final test result

The selected champion is **XGBoost — base** using the **base** feature set. Its operating threshold, selected on validation data by maximizing F2, is **0.265**.

Held-out test results:

- Accuracy: **98.2%**
- Precision: **95.2%**
- Recall: **93.7%**
- F1: **94.4%**
- F2: **94.0%**
- ROC-AUC: **0.9839**
- Average precision: **0.9679**
- Brier score: **0.0146**

Confusion-matrix counts: **1982 true negatives, 19 false positives, 25 false negatives, and 373 true positives**.

Reproduction status: **Fresh-kernel rerun passed on 2026-07-30**
